In [2]:
"""The purpose of this notebook is to transform the cleaned customer dataset into a machine learning-ready format.

- Preparing features and target variables
- Handling numerical and categorical variables
- Encoding categorical features
- Building a reusable preprocessing pipeline
- Splitting the dataset into training and testing sets
- Saving the processed data for model training

By the end of this notebook, the dataset will be fully prepared for machine learning while maintaining a scalable preprocsuitable for future customer predictions.essing workflow"""

'The purpose of this notebook is to transform the cleaned customer dataset into a machine learning-ready format.\n\n- Preparing features and target variables\n- Handling numerical and categorical variables\n- Encoding categorical features\n- Building a reusable preprocessing pipeline\n- Splitting the dataset into training and testing sets\n- Saving the processed data for model training\n\nBy the end of this notebook, the dataset will be fully prepared for machine learning while maintaining a scalable preprocsuitable for future customer predictions.essing workflow'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd

# Load the cleaned dataset prepared in Notebook 02
model_df = pd.read_csv(
    "/content/drive/MyDrive/AI-Relationship-Manager/data/processed/cleaned_customer_data.csv"
)

print(model_df.shape)
model_df.head()

(7043, 20)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes


In [8]:
X = model_df.drop("Churn Label", axis=1)
y = model_df["Churn Label"]

y = y.map({
    "No": 0,
    "Yes": 1
})

In [9]:
print("Feature Matrix Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Matrix Shape : (7043, 19)
Target Shape : (7043,)


In [10]:
print(y.value_counts())

Churn Label
0    5174
1    1869
Name: count, dtype: int64


In [11]:
# Numerical columns are continuous values
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Categorical columns contain text labels
categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features")
print(numerical_columns)

print("\nCategorical Features")
print(categorical_columns)

Numerical Features
['Tenure Months', 'Monthly Charges', 'Total Charges']

Categorical Features
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


In [12]:
from sklearn.model_selection import train_test_split

In [13]:
# 80% of the data will be used for learning.
# 20% will remain completely unseen until evaluation.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [14]:
print("Training Features :", X_train.shape)
print("Testing Features :", X_test.shape)

print()

print("Training Labels :", y_train.shape)
print("Testing Labels :", y_test.shape)

Training Features : (5634, 19)
Testing Features : (1409, 19)

Training Labels : (5634,)
Testing Labels : (1409,)


In [15]:
print("Training Dataset")

print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Dataset")

print(y_test.value_counts(normalize=True) * 100)

Training Dataset
Churn Label
0    73.464679
1    26.535321
Name: proportion, dtype: float64

Testing Dataset
Churn Label
0    73.456352
1    26.543648
Name: proportion, dtype: float64


In [16]:
# Import preprocessing tools

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [17]:
# Numerical preprocessing, only need scaling.

numerical_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

print("Numerical pipeline created successfully.")

Numerical pipeline created successfully.


In [18]:
# Convert text categories into machine-readable numbers.

categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

print("Categorical pipeline created successfully.")

Categorical pipeline created successfully.


In [19]:
preprocessor = ColumnTransformer(

    transformers=[

        # Scale numerical features
        (
            "numerical",
            numerical_pipeline,
            numerical_columns
        ),

        # Encode categorical features
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        )

    ]

)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [22]:
preprocessor.fit(X_train)

print("Preprocessor fitted successfully!")

Preprocessor fitted successfully!


In [23]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [24]:
print("Processed Training Shape :", X_train_processed.shape)
print("Processed Testing Shape :", X_test_processed.shape)

Processed Training Shape : (5634, 46)
Processed Testing Shape : (1409, 46)


In [25]:
# The transformed data should now be completely numerical.

print(type(X_train_processed))

<class 'numpy.ndarray'>


In [26]:
#Number of features after preprocessing

print(
    "Total Features After Encoding :",
    X_train_processed.shape[1]
)

Total Features After Encoding : 46


In [27]:
"""Saving the Preprocessing Pipeline, The preprocessing pipeline is saved so that the exact same transformations can be applied to future customer data."""

'Saving the Preprocessing Pipeline, The preprocessing pipeline is saved so that the exact same transformations can be applied to future customer data.'

In [28]:
import joblib

In [29]:

joblib.dump(
    preprocessor,
    "/content/drive/MyDrive/AI-Relationship-Manager/models/preprocessor.pkl"
)

print("Preprocessor saved successfully!")

Preprocessor saved successfully!


In [30]:
import numpy as np

# Save processed training features
np.save(
    "/content/drive/MyDrive/AI-Relationship-Manager/data/processed/X_train.npy",
    X_train_processed
)

# Save processed testing features
np.save(
    "/content/drive/MyDrive/AI-Relationship-Manager/data/processed/X_test.npy",
    X_test_processed
)

# Save labels
np.save(
    "/content/drive/MyDrive/AI-Relationship-Manager/data/processed/y_train.npy",
    y_train
)

np.save(
    "/content/drive/MyDrive/AI-Relationship-Manager/data/processed/y_test.npy",
    y_test
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!
